# LabelMe标注数据处理工具

## 功能概述

本Notebook实现了LabelMe标注数据的完整处理流程，包括：

1. **标注文件清洗** — 验证JSON完整性，过滤无效数据，识别重复标注
2. **数据统计分析** — 统计类别分布、标注数量，生成可视化报告
3. **样本均衡化选择** — 支持n张图片模式和n个标签样本模式的均衡采样
4. **数据格式转换** — 转换为Unsloth框架兼容格式，生成训练/验证/测试集

## 数据处理流程

```
LabelMe JSON → [1]清洗 → [2]统计 → [3]选择 → [4]转换 → 输出数据集
```

> 所有功能已封装在 `tools` 模块，支持多线程并行处理和tqdm进度条

In [1]:
import logging
import sys
import time
from pathlib import Path

_nb_file = globals().get("__vsc_ipynb_file__", "")
_bootstrap_starts = [Path.cwd()] + ([Path(_nb_file).parent] if _nb_file else [])
for _start in _bootstrap_starts:
    for _candidate in [_start] + list(_start.parents):
        if (_candidate / "pyproject.toml").exists():
            if str(_candidate) not in sys.path:
                sys.path.insert(0, str(_candidate))
            break

from unsloth_finetune.notebooking.common import initialize_notebook_context

NOTEBOOK_CONTEXT = initialize_notebook_context(notebook_file=_nb_file, cwd=Path.cwd())
NOTEBOOK_DIR = NOTEBOOK_CONTEXT["NOTEBOOK_DIR"]
PROJECT_ROOT = NOTEBOOK_CONTEXT["PROJECT_ROOT"]

from unsloth_finetune.data.labelme import (
    IN_NOTEBOOK,
    TQDM_AVAILABLE,
    BalancedSelectionResult,
    CleaningResult,
    ConversionResult,
    LabelStatistics,
    SelectionMode,
    clean_labelme_data,
    convert_to_unsloth_format,
    create_progress_bar,
    select_balanced_samples,
    statistics_labelme_labels,
)
from unsloth_finetune.data.labelme.file_utils import (
    ORJSON_AVAILABLE,
    json_dumps_str,
    json_loads,
    parse_json_file,
    write_json_file,
)

try:
    from IPython.display import HTML, display

    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False
    display = print
    HTML = str

try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.font_manager as fm
    import matplotlib.pyplot as plt

    from unsloth_finetune.notebooking.vision_shared import configure_matplotlib_for_chinese
    configure_matplotlib_for_chinese(plt, fm, warn=True, auto_download=True)

    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False
    plt = None

try:
    from PIL import Image

    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False


class NotebookUI:
    """Notebook界面美化工具 — 提供统一的步骤追踪、配置展示和结果汇总"""

    STEP_INFO = [
        ("1", "标注清洗", "验证完整性、过滤无效数据、识别重复标注"),
        ("2", "统计分析", "类别分布、标注数量统计、可视化报告"),
        ("3", "均衡选择", "基于类别的均衡采样（两种模式可选）"),
        ("4", "格式转换", "Unsloth格式转换、训练/验证/测试集划分"),
        ("5", "输出验证", "数据格式完整性验证"),
    ]

    _STYLES = {
        "step_header": "background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);padding:16px;border-radius:10px;color:white;margin:10px 0;",
        "success_card": "background:#d4edda;border:2px solid #c3e6cb;padding:16px;border-radius:10px;margin:10px 0;",
        "error_card": "background:#f8d7da;border:2px solid #f5c6cb;padding:16px;border-radius:10px;margin:10px 0;",
        "config_card": "background:#f8f9fa;padding:16px;border-radius:10px;margin:10px 0;border:1px solid #dee2e6;",
        "info_card": "background:#e7f3ff;padding:12px;border-radius:8px;margin:8px 0;border:1px solid #b8daff;",
        "warning_card": "background:#fff3cd;border:2px solid #ffeeba;padding:12px;border-radius:8px;margin:8px 0;",
    }

    def __init__(self):
        self.completed_steps = []
        self.start_time = time.time()

    def _html(self, content):
        if IPYTHON_AVAILABLE:
            display(HTML(content))
        else:
            print(content)

    def show_step_header(self, step_num, step_name, description):
        elapsed = time.time() - self.start_time
        progress_pct = len(self.completed_steps) / len(self.STEP_INFO) * 100
        markers = ""
        for sid, sname, _ in self.STEP_INFO:
            if sid in self.completed_steps:
                markers += f"<span style='color:#00ff88;font-weight:bold'>✅{sid}</span> "
            elif sid == str(step_num):
                markers += f"<span style='color:#fff;font-weight:bold'>🔄{sid}</span> "
            else:
                markers += f"<span style='color:rgba(255,255,255,0.5)'>⏳{sid}</span> "
        self._html(
            f"""<div style="{self._STYLES['step_header']}">
            <div style="font-size:18px;font-weight:bold;margin-bottom:8px">🔄 步骤 {step_num}: {step_name}</div>
            <div style="font-size:13px;color:rgba(255,255,255,0.9);margin-bottom:12px">{description}</div>
            <div style="background:rgba(255,255,255,0.3);border-radius:5px;height:24px">
                <div style="background:#00ff88;border-radius:5px;height:100%;width:{progress_pct:.0f}%;display:flex;align-items:center;justify-content:center;font-weight:bold;color:#333;font-size:11px">{progress_pct:.0f}%</div>
            </div>
            <div style="margin-top:8px;font-size:12px">{markers}</div>
            <div style="font-size:12px;margin-top:6px">⏱️ 已用时 {elapsed:.1f}秒</div>
        </div>"""
        )

    def mark_step_completed(self, step_num):
        self.completed_steps.append(str(step_num))
        elapsed = time.time() - self.start_time
        self._html(
            f"""<div style="{self._STYLES['success_card']}">
            <span style="font-size:16px;font-weight:bold;color:#155724">✅ 步骤 {step_num} 完成</span>
            <span style="font-size:12px;color:#2d5a2d;margin-left:8px">⏱️ {elapsed:.1f}秒</span>
        </div>"""
        )

    def show_config(self, config):
        dedup = "✅ 开启" if config.deduplicate else "❌ 关闭"
        img_val = "✅ 开启" if config.validate_images else "❌ 关闭"
        rows = f"""<tr style="background:#e9ecef"><th style="padding:10px;text-align:left;border:1px solid #dee2e6">参数</th><th style="padding:10px;text-align:left;border:1px solid #dee2e6">值</th></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">源目录</td><td style="padding:8px;border:1px solid #dee2e6">{config.source_dir}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">清洗输出</td><td style="padding:8px;border:1px solid #dee2e6">{config.cleaned_dir}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">统计输出</td><td style="padding:8px;border:1px solid #dee2e6">{config.stats_output}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">选择模式</td><td style="padding:8px;border:1px solid #dee2e6">{config.selection_mode}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">目标数量</td><td style="padding:8px;border:1px solid #dee2e6">{config.target_count}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">训练/验证/测试</td><td style="padding:8px;border:1px solid #dee2e6">{config.split}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">去重模式</td><td style="padding:8px;border:1px solid #dee2e6">{dedup}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">图片验证</td><td style="padding:8px;border:1px solid #dee2e6">{img_val}</td></tr>
        <tr><td style="padding:8px;border:1px solid #dee2e6;font-weight:bold">并行线程</td><td style="padding:8px;border:1px solid #dee2e6">{config.max_workers}</td></tr>"""
        self._html(
            f"""<div style="{self._STYLES['config_card']}">
            <h3 style="color:#495057;margin-bottom:12px">⚙️ 配置参数</h3>
            <table style="width:100%;border-collapse:collapse">{rows}</table>
        </div>"""
        )

    def show_result_summary(self, title, results, success=True):
        bg = self._STYLES["success_card"] if success else self._STYLES["error_card"]
        icon = "✅" if success else "⚠️"
        border_color = "#c3e6cb" if success else "#f5c6cb"
        rows = ""
        for key, value in results.items():
            rows += f"<tr><td style='padding:8px;border:1px solid {border_color};font-weight:bold'>{key}</td><td style='padding:8px;border:1px solid {border_color}'>{value}</td></tr>"
        self._html(
            f"""<div style="{bg}">
            <h4 style="margin-bottom:10px">{icon} {title}</h4>
            <table style="width:100%;border-collapse:collapse">{rows}</table>
        </div>"""
        )

    def show_error(self, step_name, error_msg):
        self._html(
            f"""<div style="{self._STYLES['error_card']}">
            <h4 style="color:#721c24;margin-bottom:8px">❌ {step_name} 执行失败</h4>
            <div style="color:#721c24;font-size:14px">{error_msg}</div>
            <div style="color:#856404;font-size:12px;margin-top:8px">💡 请检查配置参数和数据路径后重试</div>
        </div>"""
        )

    def show_info(self, message):
        self._html(
            f"""<div style="{self._STYLES['info_card']}">
            <span style="color:#004085">ℹ️ {message}</span>
        </div>"""
        )

    def show_warning(self, message):
        self._html(
            f"""<div style="{self._STYLES['warning_card']}">
            <span style="color:#856404">⚠️ {message}</span>
        </div>"""
        )

    def show_final_summary(self, cleaning_result, stats_result, selection_result, conversion_result, config):
        elapsed = time.time() - self.start_time
        items = [
            (
                "清洗合规文件",
                f"{cleaning_result.valid_count}/{cleaning_result.total_files}",
            ),
            ("类别总数", str(stats_result.total_labels)),
            ("标注实例", str(stats_result.total_label_instances)),
            ("唯一选择图片", str(selection_result.unique_image_count)),
            ("转换记录数", str(conversion_result.converted_count)),
        ]
        if conversion_result.train_split:
            items.append(("训练集", f"{conversion_result.train_split.total_records} 条"))
        if conversion_result.val_split:
            items.append(("验证集", f"{conversion_result.val_split.total_records} 条"))
        if conversion_result.test_split:
            items.append(("测试集", f"{conversion_result.test_split.total_records} 条"))
        items.append(("总耗时", f"{elapsed:.1f} 秒"))

        rows = ""
        for key, val in items:
            rows += f"<tr><td style='padding:10px;border:1px solid #dee2e6;font-weight:bold'>{key}</td><td style='padding:10px;border:1px solid #dee2e6'>{val}</td></tr>"

        output_items = [
            ("清洗目录", config.cleaned_dir),
            ("清洗报告", cleaning_result.report_path or "N/A"),
            ("统计文件", config.stats_output),
        ]
        if conversion_result.train_split:
            output_items.append(("训练集", conversion_result.train_split.output_path))
        if conversion_result.val_split:
            output_items.append(("验证集", conversion_result.val_split.output_path))
        if conversion_result.test_split:
            output_items.append(("测试集", conversion_result.test_split.output_path))

        out_rows = ""
        for key, val in output_items:
            out_rows += f"<tr><td style='padding:10px;border:1px solid #dee2e6;font-weight:bold'>{key}</td><td style='padding:10px;border:1px solid #dee2e6;font-size:12px'>{val}</td></tr>"

        self._html(
            f"""<div style="background:linear-gradient(135deg,#28a745,#1a8060);padding:20px;border-radius:12px;color:white;margin:10px 0">
            <h3 style="margin-bottom:15px">🎉 数据处理流程全部完成</h3>
            <div style="background:white;color:#333;border-radius:8px;padding:12px;margin-bottom:12px">
                <h4 style="margin-bottom:8px;color:#155724">处理统计</h4>
                <table style="width:100%;border-collapse:collapse">{rows}</table>
            </div>
            <div style="background:white;color:#333;border-radius:8px;padding:12px">
                <h4 style="margin-bottom:8px;color:#495057">输出文件</h4>
                <table style="width:100%;border-collapse:collapse">{out_rows}</table>
            </div>
        </div>"""
        )


ui = NotebookUI()
ui.show_info(
    "模块加载完成 — 环境: "
    + ("Jupyter Notebook" if IN_NOTEBOOK else "标准Python")
    + " | tqdm: "
    + ("✅" if TQDM_AVAILABLE else "❌")
    + " | matplotlib: "
    + ("✅" if MATPLOTLIB_AVAILABLE else "❌")
    + " | PIL: "
    + ("✅" if PIL_AVAILABLE else "❌")
)

### 项目路径与全局配置

修改以下配置单元格中的路径和参数即可适配不同环境。所有路径基于 `PROJECT_ROOT` 自动推导，无需手动调整其他单元格。

In [2]:
# ============================================================
# 项目路径与全局配置
# ============================================================
# 【重要】修改以下路径参数即可适配不同环境，无需改动其他单元格

from pathlib import Path

NOTEBOOK_DIR = NOTEBOOK_CONTEXT["NOTEBOOK_DIR"]
PROJECT_ROOT = NOTEBOOK_CONTEXT["PROJECT_ROOT"]

# ---------- 数据路径配置 ----------
# LabelMe原始标注数据目录 (存放.json标注文件和对应图片)
SOURCE_DIR = Path("/raid5/sh/data/wgang_40")

# 清洗后数据输出目录
CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "labelme_cleaned-wgang_40"

# 数据统计分析结果输出路径
STATS_OUTPUT = PROJECT_ROOT / "data" / "processed" / "labelme_stats-wgang_40.json"

# 均衡选择后数据输出目录
SELECTED_DIR = PROJECT_ROOT / "data" / "processed" / "labelme_selected-wgang_40"

# Unsloth格式训练数据输出目录 (后续02-微调Notebook将引用此路径下的jsonl文件)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "unsloth_training_data-wgang_40"

# 处理日志文件路径
LOG_FILE = PROJECT_ROOT / "data" / "processed" / "processing_log-wgang_40.txt"

# ---------- 数据处理参数 ----------
SELECTION_MODE = "n_images"  # 样本选择模式: "n_images"(按图片数) 或 "n_labels"(按标签数)
TARGET_COUNT = 20  # 每类目标选择数量 (仅在选择模式下生效)
RANDOM_SEED = 42  # 随机种子, 确保可复现性

# ---------- 数据集划分 ----------
SPLIT_RATIO = "8:1:1"  # 划分比例 (如 8:1:1 表示 80%训练 10%验证 10%测试)
SPLIT_METHOD = "random"  # 划分方法: "random"/"sequential"/"stratified"

# ---------- 坐标管道 ----------
COORD_NORM = "norm_1000"  # 坐标归一化模式: "raw"/"norm_1"/"norm_100"/"norm_1000"
COORD_FORMAT = "xyxy"  # 坐标格式: "xyxy"/"xywh"/"cxcywh"

# ---------- 生成策略 ----------
GEN_STRATEGY = "all_in_one"  # 生成策略: "all_in_one"/"per_class"/"both"

# ---------- Prompt模板 ----------
LANG = "zh"  # Prompt语言: "en"/"zh"
PROMPT_STYLE = "descriptive"  # Prompt风格: "simple"/"descriptive"/"cot"
PROMPT_TEMPLATE_FILE = None  # 自定义Prompt模板YAML文件路径 (None=使用内置模板)

# ---------- 输出格式 ----------
OUTPUT_SCHEMA = "openai_messages"  # 输出格式: "openai_messages"/"sharegpt"
OUTPUT_FORMAT = "box_2d_json"  # 响应内容格式: "labelme_text"/"box_2d_json"
IMAGE_PATH_MODE = "absolute"  # 图片路径模式: "relative"/"absolute"/"base64"/"copy"
COPY_IMAGES = True  # 是否复制图片到输出目录

# ---------- 数据过滤 ----------
CLASS_WHITELIST = None  # 类别白名单 (如 ["cat", "dog"])
CLASS_BLACKLIST = None  # 类别黑名单 (如 ["background", "ignore"])
CLASS_REMAP = None  # 类别重映射字典 (如 {"human": "person"})
CLASS_REMAP_FILE = None  # 类别重映射JSON文件路径
SHAPE_TYPES = ["rectangle", "polygon"]  # 要处理的形状类型
MIN_BBOX_SIZE = 2  # 最小bbox尺寸阈值
KEEP_EMPTY = False  # 是否保留空标注图片(用于负样本训练)

# ---------- 运行参数 ----------
MAX_WORKERS = 4  # 并行处理线程数
VERBOSE = True  # 是否显示详细处理信息
VALIDATE_IMAGES = True  # 是否验证图片完整性
DEDUPLICATE = True  # 是否对标注进行去重处理
PRESERVE_STRUCTURE = True  # 是否保留原始目录结构

# ---------- imageUrl处理参数 ----------
DOWNLOAD_REMOTE_IMAGES = False  # 是否下载远程图片（imageUrl字段）
REMOTE_IMAGE_DOWNLOAD_DIR = PROJECT_ROOT / "data" / "processed" / "downloaded_images-wgang_40"

# ========== 类别映射参数（在第二部分配置）==========
# 请先运行「第一部分：原始数据预扫描」查看label统计结果后，
# 在「第二部分：类别映射配置」单元格中设置 LABEL_MAPPING

print(f"项目根目录: {PROJECT_ROOT}")
print(f"原始数据目录: {SOURCE_DIR}")
print(f"输出数据目录: {OUTPUT_DIR}")
print(f"坐标归一化: {COORD_NORM}, 坐标格式: {COORD_FORMAT}")
print(f"生成策略: {GEN_STRATEGY}, 语言: {LANG}, 风格: {PROMPT_STYLE}")


项目根目录: /raid5/sh/code/unsloth-finetune
原始数据目录: /raid5/sh/data/wgang_40
输出数据目录: /raid5/sh/code/unsloth-finetune/data/processed/unsloth_training_data-wgang_40
坐标归一化: norm_1000, 坐标格式: xyxy
生成策略: all_in_one, 语言: zh, 风格: descriptive


### 配置数据路径和处理参数

请根据实际数据路径修改 `DataProcessingConfig` 中的参数。

In [3]:
class DataProcessingConfig:
    """数据处理配置: 所有路径和参数均从上方全局配置单元格获取"""

    def __init__(self):
        self.source_dir = str(SOURCE_DIR)
        self.cleaned_dir = str(CLEANED_DIR)
        self.stats_output = str(STATS_OUTPUT)
        self.selected_dir = str(SELECTED_DIR)
        self.output_dir = str(OUTPUT_DIR)
        self.log_file = str(LOG_FILE)

        self.format_subdir_name = "adv_label"
        self.cleaned_subdir_name = "cleaned_data"

        self.stats_source_dir = str(Path(CLEANED_DIR) / self.format_subdir_name)

        self.selection_mode = SELECTION_MODE
        self.target_count = TARGET_COUNT
        self.random_seed = RANDOM_SEED

        # 坐标管道
        self.coord_norm = COORD_NORM
        self.coord_format = COORD_FORMAT

        # 生成策略
        self.gen_strategy = GEN_STRATEGY

        # Prompt模板
        self.lang = LANG
        self.prompt_style = PROMPT_STYLE
        self.prompt_template_file = PROMPT_TEMPLATE_FILE

        # 划分
        self.split = SPLIT_RATIO
        self.split_method = SPLIT_METHOD

        # 输出格式
        self.output_schema = OUTPUT_SCHEMA
        self.output_format = OUTPUT_FORMAT
        self.image_path_mode = IMAGE_PATH_MODE
        self.copy_images = COPY_IMAGES

        # 数据过滤
        self.class_whitelist = CLASS_WHITELIST
        self.class_blacklist = CLASS_BLACKLIST
        self.class_remap = CLASS_REMAP
        self.class_remap_file = CLASS_REMAP_FILE
        self.shape_types = SHAPE_TYPES
        self.min_bbox_size = MIN_BBOX_SIZE
        self.keep_empty = KEEP_EMPTY

        self.validate_images = VALIDATE_IMAGES
        self.deduplicate = DEDUPLICATE
        self.preserve_structure = PRESERVE_STRUCTURE

        self.download_remote_images = DOWNLOAD_REMOTE_IMAGES
        self.remote_image_download_dir = str(REMOTE_IMAGE_DOWNLOAD_DIR)

        self.label_mapping = {}  # 将在第二部分设置

        self.max_workers = MAX_WORKERS
        self.verbose = VERBOSE


config = DataProcessingConfig()
ui.show_config(config)


参数,值
源目录,/raid5/sh/data/wgang_40
清洗输出,/raid5/sh/code/unsloth-finetune/data/processed/labelme_cleaned-wgang_40
统计输出,/raid5/sh/code/unsloth-finetune/data/processed/labelme_stats-wgang_40.json
选择模式,n_images
目标数量,20
训练/验证/测试,8:1:1
去重模式,✅ 开启
图片验证,✅ 开启
并行线程,4


## 第一部分：原始数据Label预扫描

在执行清洗之前，先扫描原始数据目录，统计所有label的出现频次。

**目的：**
- 了解原始数据中的类别分布
- 为第二部分的类别映射配置提供参考依据
- 发现潜在的类别命名不规范问题

In [4]:
ui.show_step_header(0.5, "Label预扫描", "扫描原始数据，统计所有label频次")

from unsloth_finetune.notebooking.data_prep_shared import prescan_label_statistics

prescan_result = prescan_label_statistics(Path(SOURCE_DIR), MAX_WORKERS)

ui.show_result_summary(
    "预扫描结果",
    {
        "JSON文件总数": prescan_result["total_json_files"],
        "成功扫描": prescan_result["scanned_files"],
        "跳过文件": prescan_result["skipped_files"],
        "imageUrl文件": prescan_result["image_url_files"],
        "本地图片文件": prescan_result["local_image_files"],
        "类别总数": prescan_result["total_labels"],
        "标注实例": prescan_result["total_instances"],
    },
)

print("\n" + "=" * 70)
print("所有Label频次统计（按频次排序）")
print("=" * 70)
max_label_len = max(len(str(l)) for l in prescan_result["label_counts"].keys()) if prescan_result["label_counts"] else 10
for label, count in prescan_result["label_counts"].items():
    print(f"  {str(label):<{max_label_len}} : {count:>6} 次")

print("\n" + "=" * 70)
print("📋 默认LABEL_MAPPING字典（可直接复制修改）")
print("=" * 70)
default_mapping = {label: label for label in prescan_result["label_counts"].keys()}
print("LABEL_MAPPING = {")
for label in prescan_result["label_counts"].keys():
    print(f'    "{label}": "{label}",')
print("}")


Label预扫描:   0%|          | 0/26395 [00:00<?, ?文件/s]

JSON文件总数,26395
成功扫描,26395
跳过文件,0
imageUrl文件,0
本地图片文件,26395
类别总数,42
标注实例,45615



所有Label频次统计（按频次排序）
  iron awning                     :   7646 次
  waste pile                      :   6193 次
  damaged greenhouse              :   5959 次
  pipe                            :   5164 次
  rural house under construction  :   4826 次
  gravel pile                     :   2559 次
  bricks pile                     :   2267 次
  iInflatable arches              :   1994 次
  satellite dish receiver         :   1138 次
  covered truck                   :    857 次
  highway billboard               :    719 次
  passenger ship                  :    579 次
  empty truck                     :    552 次
  uncovered truck                 :    537 次
  cargo ship                      :    508 次
  wood ship                       :    501 次
  hang balloons                   :    400 次
  green algae                     :    182 次
  stacking of tires               :    173 次
  road water logging              :    173 次
  tanker                          :    155 次
  soil coverage net               :

In [5]:
default_mapping = {
    "iron awning": "铁皮棚",
    "waste pile": "暴露的垃圾",
    "damaged greenhouse": "破损的大棚",
    "pipe": "堆积物",
    "rural house under construction": "建造中的农房",
    "gravel pile": "砂石堆",
    "bricks pile": "砖堆",
    "iInflatable arches": "气模拱门",
    "satellite dish receiver": "卫星锅",
    "covered truck": "覆盖防尘网的满载渣土车",
    "highway billboard": "高速公路上的广告牌",
    "passenger ship": "客船",
    "empty truck": "空载的渣土车",
    "uncovered truck": "未覆盖防尘网的满载渣土车",
    "cargo ship": "货船",
    "wood ship": "木船",
    "hang balloons": "悬挂气球",
    "green algae": "水面绿藻",
    "stacking of tires": "堆放一起的废旧轮胎",
    "road water logging": "道路积水",
    "tanker": "罐车",
    "soil coverage net": "裸土覆盖网",
    "muck": "渣土",
    "small ship": "小船",
    "building exterior advertisement": "建筑立面广告牌",
    "grave": "毁林造坟",
    "building damage": "破损的房屋",
    "trace of straw burning": "秸秆焚烧痕迹",
    "glass roof": "玻璃棚",
    "mobile vendors": "流动摊贩",
    "bus": "公共汽车",
    "the crops have lodged": "庄稼倒伏",
    "sand dredger": "采砂船",
    "excavator": "挖掘机",
    "standing trees change color": "变色立木",
    "bulldozer": "铲车",
    "crane": "吊车",
    "drying grains on the road": "占道晒粮",
    "chimney emission": "烟雾排放",
    "road damage": "道路破损",
    "dring along the street": "沿街晾晒",
    "traffic blur": "交通标志性模糊",
}

## 第二部分：类别映射配置

根据第一部分的预扫描结果，在此配置类别映射字典。

**使用方法：**
1. 查看上方预扫描结果中的label列表
2. 修改下方 `LABEL_MAPPING` 字典，将原始类别映射为目标类别
3. 运行本单元格应用配置
4. 在第三部分执行清洗时将自动应用映射规则

**注意：**
- 未在映射字典中的label将保持原样
- 映射规则仅在清洗阶段生效，不修改原始数据文件

In [6]:
# ============================================================
# 类别映射字典配置
# ============================================================
# 格式: {"原始类别名": "目标类别名", ...}
# 示例: LABEL_MAPPING = {" algae": "green_algae", "trash": "garbage"}

LABEL_MAPPING = default_mapping  # 请根据预扫描结果填写映射规则

# 应用配置到DataProcessingConfig
config.label_mapping = LABEL_MAPPING

if config.label_mapping:
    rows = ""
    for original, target in config.label_mapping.items():
        count = prescan_result["label_counts"].get(original, 0)
        rows += f"<tr><td style='padding:8px;border:1px solid #dee2e6'>{original}</td><td style='padding:8px;border:1px solid #dee2e6;font-weight:bold;color:#28a745'>{target}</td><td style='padding:8px;border:1px solid #dee2e6'>{count} 次</td></tr>"
    ui._html(
        f"""<div style="{ui._STYLES['success_card']}"><h4 style='color:#155724;margin-bottom:8px'>✅ 类别映射规则已配置</h4><table style='width:100%;border-collapse:collapse'><tr><th style='padding:8px;border:1px solid #dee2e6;background:#d4edda'>原始类别</th><th style='padding:8px;border:1px solid #dee2e6;background:#d4edda'>目标类别</th><th style='padding:8px;border:1px solid #dee2e6;background:#d4edda'>频次</th></tr>{rows}</table><p style='margin-top:10px;color:#155724'>映射规则将在第三部分清洗阶段自动应用</p></div>"""
    )

    unmapped_labels = [l for l in prescan_result["label_counts"].keys() if l not in config.label_mapping]
    if unmapped_labels:
        print(f"\n⚠️ 未映射的label（{len(unmapped_labels)} 个）:")
        for label in unmapped_labels[:10]:
            count = prescan_result["label_counts"].get(label, 0)
            print(f"  {label}: {count} 次")
        if len(unmapped_labels) > 10:
            print(f"  ... 还有 {len(unmapped_labels) - 10} 个未显示")
else:
    ui.show_info("未配置类别映射字典。如需应用类别映射，请修改 LABEL_MAPPING 变量后重新运行本单元格。")

原始类别,目标类别,频次
iron awning,铁皮棚,7646 次
waste pile,暴露的垃圾,6193 次
damaged greenhouse,破损的大棚,5959 次
pipe,堆积物,5164 次
rural house under construction,建造中的农房,4826 次
gravel pile,砂石堆,2559 次
bricks pile,砖堆,2267 次
iInflatable arches,气模拱门,1994 次
satellite dish receiver,卫星锅,1138 次
covered truck,覆盖防尘网的满载渣土车,857 次


## 第三部分：数据清洗与完整性验证

使用 `LabelMeCleaner` 对原始标注数据进行清洗处理：

- **JSON完整性验证** — 检查必需字段、数据类型、几何坐标有效性
- **重复标注识别** — 基于MD5哈希识别完全重复的标注文件
- **类别映射应用** — 将原始label替换为配置的目标类别
- **imageUrl兼容性** — 支持远程图片URL，可选择下载到本地
- **多线程并行处理** — 显著提升大规模数据处理效率

### 执行清洗流程

路径和参数已在上方 **0. 项目路径与全局配置** 单元格中集中定义，类别映射已在 **第二部分** 配置。

In [ ]:
ui.show_step_header(1, "标注清洗", "验证JSON完整性，过滤无效数据，识别重复标注，应用类别映射")

cleaning_result = None
try:
    cleaning_result = clean_labelme_data(
        source_dir=config.source_dir,
        target_dir=config.cleaned_dir,
        preserve_structure=config.preserve_structure,
        copy_images=config.copy_images,
        deduplicate=config.deduplicate,
        generate_report=True,
        log_file=config.log_file,
        max_workers=config.max_workers,
        use_tqdm=True,
        label_mapping=config.label_mapping,
        format_subdir_name=config.format_subdir_name,
        cleaned_subdir_name=config.cleaned_subdir_name,
        download_remote_images=config.download_remote_images,
        remote_image_download_dir=config.remote_image_download_dir,
    )

    ui.mark_step_completed(1)
    ui.show_result_summary(
        "清洗结果",
        {
            "目标目录": config.cleaned_dir,
            "清洗数据目录": str(Path(config.cleaned_dir) / config.cleaned_subdir_name),
            "格式转换目录": str(Path(config.cleaned_dir) / config.format_subdir_name),
            "总文件数": cleaning_result.total_files,
            "合规文件": (f"{cleaning_result.valid_count} ({cleaning_result.valid_ratio:.1f}%)" if cleaning_result.valid_ratio else str(cleaning_result.valid_count)),
            "不合规文件": cleaning_result.invalid_count,
            "重复标注": cleaning_result.duplicate_count,
            "复制JSON": len(cleaning_result.copied_json_files),
            "复制图片": len(cleaning_result.copied_image_files),
            "类别映射": (f"{len(config.label_mapping)} 个规则" if config.label_mapping else "未启用"),
            "远程图片下载": "启用" if config.download_remote_images else "未启用",
            "耗时": (f"{cleaning_result.duration:.2f}秒" if cleaning_result.duration else "N/A"),
        },
    )
except Exception as e:
    ui.show_error("标注清洗", str(e))
    cleaning_result = CleaningResult()

[1/6] 数据验证:   0%|          | 0/26395 [00:00<?, ?项/s]

[2/6] 去重处理:   0%|          | 0/25922 [00:00<?, ?项/s]

[3/6] 文件复制:   0%|          | 0/25922 [00:00<?, ?项/s]

[4/6] 格式转换:   0%|          | 0/25922 [00:00<?, ?项/s]

### 查看清洗详情

显示不合规文件和重复标注的详细分类信息。

In [ ]:
if cleaning_result and cleaning_result.invalid_files:
    status_counts = {}
    for item in cleaning_result.invalid_files:
        status = item.get("status", "unknown")
        status_counts[status] = status_counts.get(status, 0) + 1

    rows = ""
    for status, count in sorted(status_counts.items(), key=lambda x: x[1], reverse=True):
        rows += f"<tr><td style='padding:6px;border:1px solid #f5c6cb;font-weight:bold'>{status}</td><td style='padding:6px;border:1px solid #f5c6cb'>{count} 个</td></tr>"
    ui._html(f"""<div style="{ui._STYLES['warning_card']}"><h4 style='color:#856404;margin-bottom:8px'>⚠️ 不合规文件分类</h4><table style='width:100%;border-collapse:collapse'>{rows}</table></div>""")

    print("\n不合规文件示例（前5个）:")
    for i, item in enumerate(cleaning_result.invalid_files[:5], 1):
        print(f"  {i}. {Path(item['file']).name} — {item['reason']}")
else:
    ui.show_info("无不合规文件 ✅")

if cleaning_result and cleaning_result.duplicate_files:
    image_counts = {}
    for item in cleaning_result.duplicate_files:
        image_file = item.get("image_file", "unknown")
        image_counts[image_file] = image_counts.get(image_file, 0) + 1

    rows = ""
    for image_file, count in sorted(image_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        rows += f"<tr><td style='padding:6px;border:1px solid #ffeeba'>{Path(image_file).name}</td><td style='padding:6px;border:1px solid #ffeeba'>{count} 个重复</td></tr>"
    ui._html(
        f"""<div style="{ui._STYLES['warning_card']}"><h4 style='color:#856404;margin-bottom:8px'>⚠️ 重复标注文件（Top 5）</h4><table style='width:100%;border-collapse:collapse'>{rows}</table></div>"""
    )
else:
    ui.show_info("无重复标注文件 ✅")

## 第四部分：数据统计分析

使用 `LabelMeLabelStatistics` 对清洗后的数据进行类别统计：

- **总文件数/处理/跳过统计** — 含无imageUrl和解析错误的分类计数
- **类别总数/标注实例** — 每个label的文件数、实例数、单文件标注范围
- **多线程并行处理** — `max_workers` 控制并行线程数
- **结构化JSON输出** — 类别按字母排序，数量键按数字排序

In [ ]:
ui.show_step_header(2, "统计分析", "类别分布、标注数量统计、可视化报告（基于格式转换后的数据）")

stats_result = None
try:
    stats_result = statistics_labelme_labels(
        source_dir=config.stats_source_dir,
        recursive=True,
        use_relative_path=True,
        max_workers=config.max_workers,
        log_file=config.log_file,
        use_tqdm=True,
    )

    stats_path = Path(config.stats_output)
    write_json_file(stats_path, stats_result.to_structured_dict(), indent=2)

    ui.mark_step_completed(2)
    ui.show_result_summary(
        "统计结果",
        {
            "统计源目录": config.stats_source_dir,
            "总JSON文件": stats_result.total_json_files,
            "有效处理": stats_result.processed_files,
            "跳过(无imageUrl)": stats_result.skipped_no_imageurl,
            "跳过(解析错误)": stats_result.skipped_parse_error,
            "类别总数": stats_result.total_labels,
            "标注实例": stats_result.total_label_instances,
            "统计文件": config.stats_output,
        },
    )
except Exception as e:
    ui.show_error("统计分析", str(e))
    stats_result = LabelStatistics(source_dir=config.stats_source_dir)

### 类别分布可视化

生成Top 15类别标注实例和文件分布的双栏对比图表。

In [ ]:
if MATPLOTLIB_AVAILABLE and stats_result and stats_result.label_counts:
    summary = stats_result.get_label_summary()
    sorted_summary = sorted(summary.items(), key=lambda x: x[1]["total_instances"], reverse=True)
    top_n = min(15, len(sorted_summary))

    labels = [item[0] for item in sorted_summary[:top_n]]
    instances = [item[1]["total_instances"] for item in sorted_summary[:top_n]]
    files = [item[1]["total_files"] for item in sorted_summary[:top_n]]

    plt.rcParams.update({"font.size": 11, "figure.dpi": 120})
    fig, axes = plt.subplots(1, 2, figsize=(15, max(5, top_n * 0.35)))

    colors_instances = plt.cm.Blues([(0.4 + 0.6 * i / top_n) for i in range(top_n)])
    colors_files = plt.cm.Oranges([(0.4 + 0.6 * i / top_n) for i in range(top_n)])

    axes[0].barh(
        range(top_n),
        instances,
        color=colors_instances,
        edgecolor="white",
        linewidth=0.5,
    )
    axes[0].set_yticks(range(top_n))
    axes[0].set_yticklabels(labels, fontsize=10)
    axes[0].invert_yaxis()
    axes[0].set_xlabel("标注实例数", fontsize=12)
    axes[0].set_title("类别标注实例分布（Top 15）", fontsize=13, fontweight="bold")
    axes[0].grid(axis="x", alpha=0.3, linestyle="--")
    for i, v in enumerate(instances):
        axes[0].text(v + 0.3, i, str(v), va="center", fontsize=9, color="#333")

    axes[1].barh(range(top_n), files, color=colors_files, edgecolor="white", linewidth=0.5)
    axes[1].set_yticks(range(top_n))
    axes[1].set_yticklabels(labels, fontsize=10)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("文件数", fontsize=12)
    axes[1].set_title("类别文件分布（Top 15）", fontsize=13, fontweight="bold")
    axes[1].grid(axis="x", alpha=0.3, linestyle="--")
    for i, v in enumerate(files):
        axes[1].text(v + 0.3, i, str(v), va="center", fontsize=9, color="#333")

    plt.tight_layout(pad=2.0)
    chart_path = str(Path(config.output_dir) / "label_distribution_chart.png")
    Path(chart_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(chart_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close()
    ui.show_info(f"类别分布图表已保存到: {chart_path}")
else:
    ui.show_warning("无法生成图表 — matplotlib未安装或无统计数据")

if stats_result and stats_result.label_counts:
    summary = stats_result.get_label_summary()
    sorted_summary = sorted(summary.items(), key=lambda x: x[1]["total_instances"], reverse=True)
    print("\n类别详情（Top 10）:")
    for label, info in sorted_summary[:10]:
        print(f"  {label}: {info['total_instances']}实例 / {info['total_files']}文件 / 范围[{info['min_per_file']}~{info['max_per_file']}]")

## 第五部分：样本均衡化选择

基于类别分布的样本挑选机制，支持两种模式：

**n张图片模式 (`n_images`):**
- 类别图片 > n: 按标注数量降序选取前n张不重复图片
- 类别图片 ≤ n: 循环选取直至达到n张（允许重复）

**n个标签样本模式 (`n_labels`):**
- 类别总样本 > n: 按标注数量降序依次选取至累计≥n
- 类别总样本 ≤ n: 循环选取至累计≥n（允许重复）

In [ ]:
ui.show_step_header(3, "均衡选择", f"模式: {config.selection_mode}, 目标: {config.target_count}")

selection_result = None
try:
    selection_result = select_balanced_samples(
        source_dir=config.stats_source_dir,
        mode=config.selection_mode,
        target_count=config.target_count,
        random_seed=config.random_seed,
        validate_images=config.validate_images,
        log_file=config.log_file,
        max_workers=config.max_workers,
        use_tqdm=True,
    )

    selection_output = str(Path(config.output_dir) / "selection_result.json")
    write_json_file(Path(selection_output), selection_result.to_dict(), indent=2)

    ui.mark_step_completed(3)
    ui.show_result_summary(
        "选择结果",
        {
            "类别总数": len(selection_result.category_results),
            "总选择图片": selection_result.total_selected_images,
            "唯一图片": selection_result.unique_image_count,
            "结果文件": selection_output,
        },
    )
except Exception as e:
    ui.show_error("均衡选择", str(e))
    selection_result = BalancedSelectionResult(
        source_dir=config.stats_source_dir,
        mode=SelectionMode.N_IMAGES,
        target_count=config.target_count,
    )

### 各类别选择详情

In [ ]:
if selection_result and selection_result.category_results:
    rows = ""
    for category, sel in sorted(selection_result.category_results.items()):
        dup_str = f"（重复{sel.duplicate_count}）" if sel.has_duplicates else ""
        rows += f"<tr><td style='padding:6px;border:1px solid #c3e6cb;font-weight:bold'>{category}</td>"
        rows += f"<td style='padding:6px;border:1px solid #c3e6cb'>{sel.total_selected_images}{dup_str}</td>"
        rows += f"<td style='padding:6px;border:1px solid #c3e6cb'>{sel.total_selected_labels}</td>"
        rows += f"<td style='padding:6px;border:1px solid #c3e6cb'>{sel.available_images}</td></tr>"

    ui._html(
        f"""<div style="{ui._STYLES['config_card']}">
        <h4 style="color:#495057;margin-bottom:8px">📊 各类别选择详情</h4>
        <table style="width:100%;border-collapse:collapse">
            <tr style="background:#e9ecef"><th style="padding:8px;border:1px solid #dee2e6">类别</th><th style="padding:8px;border:1px solid #dee2e6">选择图片</th><th style="padding:8px;border:1px solid #dee2e6">选择标签</th><th style="padding:8px;border:1px solid #dee2e6">可用图片</th></tr>
            {rows}
        </table>
    </div>"""
    )

    if selection_result.duration:
        ui.show_info(f"选择耗时: {selection_result.duration:.2f}秒")
else:
    ui.show_warning("无选择结果数据")

## 第六部分：数据格式转换

使用 `LabelMeConverter` 将LabelMe标注转换为Unsloth框架兼容格式：

- **标注形状 → 边界框** — 支持polygon/rectangle/circle等形状类型
- **坐标归一化** — 可选归一化至[0,1]范围
- **对话格式生成** — 符合Unsloth多模态训练的messages格式
- **数据集划分** — 自动划分训练集/验证集/测试集并输出JSONL文件

### JSONL存储格式（对齐Unsloth官方标准）

JSONL文件中采用以下格式存储数据：

```json
{
  "messages": [
    {"role": "user", "content": [{"type": "text", "text": "请分析这张图像，识别并定位其中的 目标类别。"}]},
    {"role": "assistant", "content": [{"type": "text", "text": "..."}]}
  ],
  "images": ["path/to/image.jpg"],
  "metadata": {"image_width": 1024, "image_height": 1024, ...}
}
```

**关键说明：**

- `messages` 中 `user` 的 `content` 仅包含 `{"type": "text"}` — 不包含图片占位符
- 图片路径存储在独立的 `images` 字段（JSONL无法序列化PIL.Image对象）
- 训练时由 `MultimodalDataset` 加载图片并嵌入 `messages`，生成Unsloth官方格式：

```python
# 训练时的实际格式（图片嵌入messages content中）
{"messages": [
    {"role": "user", "content": [
        {"type": "image", "image": PIL.Image.Image},
        {"type": "text", "text": "请分析这张图像..."}
    ]},
    {"role": "assistant", "content": [{"type": "text", "text": "..."}]}
]}
```

此格式与Unsloth官方推荐的视觉微调数据格式完全一致（参考 `00-官方-Gemma4_(E4B)_Vision.ipynb`），配合 `UnslothVisionDataCollator` 使用。


In [ ]:
ui.show_step_header(4, "格式转换", "转换为Unsloth格式，划分训练/验证/测试集")

selected_json_files = None
if selection_result and selection_result.category_results:
    seen = set()
    selected_json_files = []
    for cat_result in selection_result.category_results.values():
        for img in cat_result.selected_images:
            p = img.json_path
            if p not in seen:
                seen.add(p)
                selected_json_files.append(p)
    ui.show_info(f"基于均衡选择结果: {len(selected_json_files)} 个唯一文件")
else:
    ui.show_warning("无均衡选择结果，将对全部文件进行转换")

conversion_result = None
try:
    conversion_result = convert_to_unsloth_format(
        source_dir=config.stats_source_dir,
        output_dir=config.output_dir,
        coord_norm=config.coord_norm,
        coord_format=config.coord_format,
        gen_strategy=config.gen_strategy,
        lang=config.lang,
        prompt_style=config.prompt_style,
        prompt_template_file=config.prompt_template_file,
        split=config.split,
        split_method=config.split_method,
        random_seed=config.random_seed,
        output_schema=config.output_schema,
        output_format=config.output_format,
        image_path_mode=config.image_path_mode,
        copy_images=config.copy_images,
        class_whitelist=config.class_whitelist,
        class_blacklist=config.class_blacklist,
        class_remap=config.class_remap,
        class_remap_file=config.class_remap_file,
        shape_types=config.shape_types,
        min_bbox_size=config.min_bbox_size,
        keep_empty=config.keep_empty,
        validate_images=config.validate_images,
        log_file=config.log_file,
        max_workers=config.max_workers,
        use_tqdm=True,
        selected_files=selected_json_files,
    )

    ui.mark_step_completed(4)
    ui.show_result_summary(
        "转换结果",
        {
            "总JSON文件": conversion_result.total_json_files,
            "成功转换": (f"{conversion_result.converted_count} ({conversion_result.conversion_rate:.1f}%)" if conversion_result.conversion_rate else str(conversion_result.converted_count)),
            "转换失败": conversion_result.failed_count,
            "跳过文件": conversion_result.skipped_count,
        },
    )
except Exception as e:
    ui.show_error("格式转换", str(e))
    conversion_result = ConversionResult(
        source_dir=config.stats_source_dir,
        output_dir=config.output_dir,
    )


### 数据集划分详情与格式验证

查看训练集/验证集/测试集的统计数据，并验证JSONL格式完整性。

In [ ]:
if conversion_result:
    split_rows = ""
    for name, split in [
        ("训练集", conversion_result.train_split),
        ("验证集", conversion_result.val_split),
        ("测试集", conversion_result.test_split),
    ]:
        if split:
            split_rows += f"<tr><td style='padding:8px;border:1px solid #c3e6cb;font-weight:bold'>{name}</td>"
            split_rows += f"<td style='padding:8px;border:1px solid #c3e6cb'>{split.total_records}</td>"
            split_rows += f"<td style='padding:8px;border:1px solid #c3e6cb'>{split.total_images}</td>"
            split_rows += f"<td style='padding:8px;border:1px solid #c3e6cb'>{split.total_objects}</td>"
            split_rows += f"<td style='padding:8px;border:1px solid #c3e6cb;font-size:11px'>{split.output_path}</td></tr>"

    if split_rows:
        ui._html(
            f"""<div style="{ui._STYLES['config_card']}">
            <h4 style="color:#495057;margin-bottom:8px">📊 数据集划分统计</h4>
            <table style="width:100%;border-collapse:collapse">
                <tr style="background:#e9ecef"><th style="padding:8px;border:1px solid #dee2e6">数据集</th><th style="padding:8px;border:1px solid #dee2e6">记录数</th><th style="padding:8px;border:1px solid #dee2e6">图片数</th><th style="padding:8px;border:1px solid #dee2e6">对象数</th><th style="padding:8px;border:1px solid #dee2e6">输出路径</th></tr>
                {split_rows}
            </table>
        </div>"""
        )

    if conversion_result.duration:
        ui.show_info(f"转换耗时: {conversion_result.duration:.2f}秒")
else:
    ui.show_warning("无转换结果数据")

In [ ]:
from unsloth_finetune.notebooking.data_prep_shared import verify_unsloth_format

ui.show_step_header(5, "输出验证", "验证数据格式完整性")

if conversion_result:
    all_passed = True
    for name, split in [
        ("训练集", conversion_result.train_split),
        ("验证集", conversion_result.val_split),
        ("测试集", conversion_result.test_split),
    ]:
        if split and split.output_path:
            passed, info = verify_unsloth_format(split.output_path)
            all_passed = all_passed and passed
            ui.show_result_summary(f"{name}格式验证", info, success=passed)

    ui.mark_step_completed(5 if all_passed else 5)
else:
    ui.show_warning("无转换结果，跳过验证")


### 查看转换示例

展示一条转换后的数据样本格式。

In [ ]:
if conversion_result and conversion_result.train_split and conversion_result.train_split.output_path:
    jsonl_path = conversion_result.train_split.output_path
    if Path(jsonl_path).exists():
        with open(jsonl_path, "r", encoding="utf-8") as f:
            line = f.readline()
            if line:
                record = json_loads(line)
                # -- 数据格式文本展示 --
                ui._html(
                    f"""<div style="{ui._STYLES['config_card']}">
                    <h4 style="color:#495057;margin-bottom:12px">📝 转换数据样本示例</h4>
                    <div style="margin-bottom:8px"><b>图像路径:</b> {record.get('images', ['N/A'])[0]}</div>
                    <div style="margin-bottom:8px"><b>用户消息:</b><br><pre style="background:#fff;padding:8px;border-radius:4px;margin:4px 0;font-size:12px;white-space:pre-wrap">{"".join(item.get("text","") for item in record["messages"][0].get("content",[]) if item.get("type")=="text")}</pre></div>
                    <div style="margin-bottom:8px"><b>助手消息:</b><br><pre style="background:#fff;padding:8px;border-radius:4px;margin:4px 0;font-size:12px;white-space:pre-wrap">{"".join(item.get("text","") for item in record["messages"][1].get("content",[]) if item.get("type")=="text")}</pre></div>
                    <div><b>元数据:</b> 尺寸={record.get("metadata",{}).get("image_width",0)}x{record.get("metadata",{}).get("image_height",0)}, 标注数={record.get("metadata",{}).get("num_objects",0)}</div>
                    <div><b>输出格式:</b> {record.get("metadata",{}).get("output_format","N/A")}</div>
                </div>"""
                )

                # -- 矩形框绘制在图片上 --
                if PIL_AVAILABLE and MATPLOTLIB_AVAILABLE:
                    import re
                    from unsloth_finetune.notebooking.vision_shared import configure_matplotlib_for_chinese
                    configure_matplotlib_for_chinese(plt, fm, warn=True, auto_download=True)

                    meta = record.get("metadata", {})
                    img_w = meta.get("image_width", 0)
                    img_h = meta.get("image_height", 0)
                    coord_format = meta.get("coord_format", config.coord_format)
                    coord_norm = meta.get("coord_norm", config.coord_norm)
                    output_format = meta.get("output_format", config.output_format)
                    assistant_text = "".join(
                        item.get("text", "")
                        for item in record["messages"][1].get("content", [])
                        if item.get("type") == "text"
                    )

                    # 坐标反变换: 归一化值 → 像素xyxy
                    def _to_pixel_xyxy(coords, c_fmt, c_norm, w, h):
                        scale_map = {"raw": 1, "norm_1": 1, "norm_100": 100, "norm_1000": 1000}
                        scale = scale_map.get(c_norm, 1)
                        if c_norm == "norm_1":
                            px = [c * d for c, d in zip(coords, [w, h, w, h])]
                        elif c_norm in ("norm_100", "norm_1000"):
                            px = [c * d / scale for c, d in zip(coords, [w, h, w, h])]
                        else:
                            px = list(coords)
                        if c_fmt == "xywh":
                            x, y, ww, hh = px
                            return [x, y, x + ww, y + hh]
                        elif c_fmt == "cxcywh":
                            cx, cy, ww, hh = px
                            return [cx - ww / 2, cy - hh / 2, cx + ww / 2, cy + hh / 2]
                        return px

                    # 解析检测框
                    detections = []
                    bbox_colors = ["#FF3838", "#FF9D97", "#FF701F", "#FFB21D", "#48F906", "#1993F0", "#6D3FCD", "#B21FC5"]
                    if output_format == "box_2d_json":
                        try:
                            items = json_loads(assistant_text)
                            for item in items:
                                box = item.get("box_2d", [])
                                label = item.get("label", "")
                                if len(box) == 4:
                                    xyxy = _to_pixel_xyxy(box, coord_format, coord_norm, img_w, img_h)
                                    detections.append((label, xyxy))
                        except Exception:
                            pass
                    else:
                        # labelme_text 格式
                        en_dash = "–"  # Unicode en-dash
                        for m in re.finditer(rf"[-{en_dash}]\s*([^:\[\]]+?)\s*:\s*\[([\d.,\s]+)\]", assistant_text):
                            label = m.group(1).strip()
                            coords = [float(x.strip()) for x in m.group(2).split(",")]
                            if len(coords) == 4:
                                xyxy = _to_pixel_xyxy(coords, coord_format, coord_norm, img_w, img_h)
                                detections.append((label, xyxy))

                    # 加载图像
                    img_path = record.get("images", [""])[0] if record.get("images") else ""
                    resolved = None
                    for candidate in [Path(img_path), Path(config.output_dir) / img_path, Path(PROJECT_ROOT) / img_path]:
                        if candidate.exists():
                            resolved = candidate
                            break

                    if resolved and detections:
                        img = Image.open(resolved)
                        fig, ax = plt.subplots(1, 1, figsize=(10, max(6, img_h / img_w * 10)))
                        ax.imshow(img)
                        for di, (label, (x1, y1, x2, y2)) in enumerate(detections):
                            color = bbox_colors[di % len(bbox_colors)]
                            rect = plt.Rectangle(
                                (x1, y1), x2 - x1, y2 - y1,
                                linewidth=2.5, edgecolor=color, facecolor="none",
                            )
                            ax.add_patch(rect)
                            ax.text(
                                x1, y1 - 3, label,
                                fontsize=11, color=color, fontweight="bold",
                                verticalalignment="bottom",
                                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor=color, linewidth=1),
                            )
                        ax.set_title(
                            f"📝 样本示例: {len(detections)} 个检测框  (格式: {coord_format}, 归一化: {coord_norm})",
                            fontsize=13,
                        )
                        ax.axis("off")
                        plt.tight_layout()
                        plt.show()
                    elif resolved and not detections:
                        img = Image.open(resolved)
                        fig, ax = plt.subplots(1, 1, figsize=(10, max(6, img_h / img_w * 10)))
                        ax.imshow(img)
                        ax.set_title("📝 样本示例 (未解析到检测框)", fontsize=13)
                        ax.axis("off")
                        plt.tight_layout()
                        plt.show()
                    elif not resolved:
                        ui.show_warning(f"图像文件未找到: {img_path} (已尝试多种路径组合)")
                elif not PIL_AVAILABLE or not MATPLOTLIB_AVAILABLE:
                    ui.show_info("PIL/matplotlib不可用, 仅展示数据格式信息(无法绘制图像)")
            else:
                ui.show_warning("训练集文件为空")
    else:
        ui.show_warning(f"文件不存在: {jsonl_path}")
else:
    ui.show_warning("无训练集数据")


### 转换数据可视化

将转换后的检测框绘制到原始图像上，直观验证坐标定位是否准确：

- 绘制矩形检测框 + 类别标签，覆盖在原图上展示
- 显示坐标反转换详情（归一化/格式 → 像素值），方便核对数据格式

In [ ]:
# ============================================================
# 转换数据可视化: 在图像上绘制检测框 + 展示数据格式详情
# ============================================================
import re

# 确保matplotlib中文字体已配置 (若跳过了Cell 1则在此处重新配置)
if MATPLOTLIB_AVAILABLE:
    from unsloth_finetune.notebooking.vision_shared import configure_matplotlib_for_chinese
    configure_matplotlib_for_chinese(plt, fm, warn=True, auto_download=True)

VIS_MAX_SAMPLES = 3  # 最多展示的样本数


def _coords_to_pixel_xyxy(coords, coord_format, coord_norm, img_w, img_h):
    """将转换后的坐标还原为像素坐标(xyxy格式), 用于绘图"""
    # 1) 反归一化
    scale_map = {"raw": 1, "norm_1": 1, "norm_100": 100, "norm_1000": 1000}
    scale = scale_map.get(coord_norm, 1)
    if coord_norm == "norm_1":
        pixel = [c * d for c, d in zip(coords, [img_w, img_h, img_w, img_h])]
    elif coord_norm in ("norm_100", "norm_1000"):
        pixel = [c * d / scale for c, d in zip(coords, [img_w, img_h, img_w, img_h])]
    else:
        pixel = list(coords)

    # 2) 格式转换到 xyxy
    if coord_format == "xywh":
        x, y, w, h = pixel
        return [x, y, x + w, y + h]
    elif coord_format == "cxcywh":
        cx, cy, w, h = pixel
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return pixel  # xyxy


def _parse_detections(assistant_text, output_format, coord_format, coord_norm, img_w, img_h):
    """从助手文本中解析检测框, 返回 [(label, [x1,y1,x2,y2])]"""
    detections = []

    if output_format == "box_2d_json":
        try:
            items = json_loads(assistant_text)
            for item in items:
                box = item.get("box_2d", [])
                label = item.get("label", "")
                if len(box) == 4:
                    xyxy = _coords_to_pixel_xyxy(box, coord_format, coord_norm, img_w, img_h)
                    detections.append((label, xyxy))
        except Exception:
            pass
    else:
        # labelme_text 格式
        # 模式1: "- label: [x1,y1,x2,y2]"
        for m in re.finditer(r"[-–]\s*([^:\[\]]+?)\s*:\s*\[([\d.,\s]+)\]", assistant_text):
            label = m.group(1).strip()
            coords = [float(x.strip()) for x in m.group(2).split(",")]
            if len(coords) == 4:
                xyxy = _coords_to_pixel_xyxy(coords, coord_format, coord_norm, img_w, img_h)
                detections.append((label, xyxy))
        if not detections:
            # 模式2: "- [x1,y1,x2,y2]" (per_class格式)
            for m in re.finditer(r"[-–]\s*\[([\d.,\s]+)\]", assistant_text):
                coords = [float(x.strip()) for x in m.group(1).split(",")]
                if len(coords) == 4:
                    xyxy = _coords_to_pixel_xyxy(coords, coord_format, coord_norm, img_w, img_h)
                    detections.append(("目标", xyxy))
    return detections


def _extract_record_fields(record, schema):
    """从JSONL记录中提取通用字段 (兼容 openai_messages / sharegpt)"""
    if schema == "sharegpt":
        img_path = record.get("image", "")
        user_text = ""
        assistant_text = ""
        for conv in record.get("conversations", []):
            if conv.get("from") == "human":
                user_text = conv.get("value", "").replace("<image>\n", "")
            elif conv.get("from") == "gpt":
                assistant_text = conv.get("value", "")
        meta = record.get("metadata", {})
    else:
        img_path = record.get("images", [""])[0] if record.get("images") else ""
        msgs = record.get("messages", [])
        user_text = "".join(
            item.get("text", "") for item in msgs[0].get("content", []) if item.get("type") == "text"
        ) if len(msgs) >= 1 else ""
        assistant_text = "".join(
            item.get("text", "") for item in msgs[1].get("content", []) if item.get("type") == "text"
        ) if len(msgs) >= 2 else ""
        meta = record.get("metadata", {})

    return {
        "img_path": img_path,
        "user_text": user_text,
        "assistant_text": assistant_text,
        "img_w": meta.get("image_width", 0),
        "img_h": meta.get("image_height", 0),
        "coord_format": meta.get("coord_format", config.coord_format),
        "coord_norm": meta.get("coord_norm", config.coord_norm),
        "output_format": meta.get("output_format", config.output_format),
        "num_objects": meta.get("num_objects", 0),
    }


def _resolve_image_path(img_path):
    """尝试多种路径策略加载图像"""
    candidates = [Path(img_path)]
    # 相对于输出目录
    candidates.append(Path(config.output_dir) / img_path)
    # 相对于项目根目录
    candidates.append(Path(PROJECT_ROOT) / img_path)
    for p in candidates:
        if p.exists():
            return p
    return None


# ---- 主逻辑 ----
if conversion_result and conversion_result.train_split and conversion_result.train_split.output_path:
    jsonl_path = conversion_result.train_split.output_path
    if not Path(jsonl_path).exists():
        ui.show_warning(f"文件不存在: {jsonl_path}")
    else:
        samples = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= VIS_MAX_SAMPLES:
                    break
                try:
                    samples.append(json_loads(line))
                except Exception:
                    continue

        if not samples:
            ui.show_warning("训练集文件为空，无可展示样本")
        else:
            schema = config.output_schema
            bbox_colors = [
                "#FF3838", "#FF9D97", "#FF701F", "#FFB21D", "#CFD011",
                "#48F906", "#1993F0", "#6D3FCD", "#B21FC5", "#F090F0",
            ]

            for idx, record in enumerate(samples):
                fields = _extract_record_fields(record, schema)
                img_path = fields["img_path"]
                img_w = fields["img_w"]
                img_h = fields["img_h"]
                coord_fmt = fields["coord_format"]
                coord_nrm = fields["coord_norm"]
                out_fmt = fields["output_format"]
                user_text = fields["user_text"]
                assistant_text = fields["assistant_text"]

                detections = _parse_detections(
                    assistant_text, out_fmt, coord_fmt, coord_nrm, img_w, img_h
                )

                # -- 数据格式详情 --
                ui._html(
                    f"""<div style="{ui._STYLES['config_card']}">
                    <h4 style="color:#495057;margin-bottom:12px">样本 {idx + 1}/{len(samples)} — 数据格式详情</h4>
                    <div style="margin-bottom:6px"><b>图像路径:</b> {img_path}</div>
                    <div style="margin-bottom:6px"><b>坐标格式:</b> {coord_fmt} | <b>归一化模式:</b> {coord_nrm} | <b>输出格式:</b> {out_fmt}</div>
                    <div style="margin-bottom:6px"><b>图像尺寸:</b> {img_w}x{img_h} | <b>检测框数:</b> {len(detections)}</div>
                    <div style="margin-bottom:6px"><b>用户消息:</b><br>
                    <pre style="background:#fff;padding:8px;border-radius:4px;font-size:12px;white-space:pre-wrap;max-height:120px;overflow:auto">{user_text[:300]}</pre></div>
                    <div style="margin-bottom:6px"><b>助手消息:</b><br>
                    <pre style="background:#fff;padding:8px;border-radius:4px;font-size:12px;white-space:pre-wrap;max-height:120px;overflow:auto">{assistant_text[:500]}</pre></div>
                </div>"""
                )

                # -- 检测框坐标反转换表 --
                if detections:
                    coord_rows = ""
                    for di, (label, xyxy) in enumerate(detections):
                        color = bbox_colors[di % len(bbox_colors)]
                        coord_rows += (
                            f"<tr><td style='padding:6px;border:1px solid #dee2e6;font-weight:bold;color:{color}'>{label}</td>"
                            f"<td style='padding:6px;border:1px solid #dee2e6'>{coord_fmt}</td>"
                            f"<td style='padding:6px;border:1px solid #dee2e6'>{coord_nrm}</td>"
                            f"<td style='padding:6px;border:1px solid #dee2e6;font-size:11px'>{int(xyxy[0])},{int(xyxy[1])},{int(xyxy[2])},{int(xyxy[3])}</td></tr>"
                        )
                    ui._html(
                        f"""<div style="{ui._STYLES['config_card']}">
                        <h4 style="color:#495057;margin-bottom:8px">检测框坐标反转换 (归一化 → 像素)</h4>
                        <table style="width:100%;border-collapse:collapse">
                            <tr style="background:#e9ecef"><th style="padding:6px;border:1px solid #dee2e6">类别</th><th style="padding:6px;border:1px solid #dee2e6">坐标格式</th><th style="padding:6px;border:1px solid #dee2e6">归一化</th><th style="padding:6px;border:1px solid #dee2e6">像素坐标(xyxy)</th></tr>
                            {coord_rows}
                        </table>
                    </div>"""
                    )

                # -- 图像 + 检测框可视化 --
                if PIL_AVAILABLE and MATPLOTLIB_AVAILABLE and img_path:
                    resolved = _resolve_image_path(img_path)
                    if resolved:
                        img = Image.open(resolved)
                        fig, ax = plt.subplots(1, 1, figsize=(10, max(6, img_h / img_w * 10)))
                        ax.imshow(img)

                        for di, (label, (x1, y1, x2, y2)) in enumerate(detections):
                            color = bbox_colors[di % len(bbox_colors)]
                            rect = plt.Rectangle(
                                (x1, y1), x2 - x1, y2 - y1,
                                linewidth=2.5, edgecolor=color, facecolor="none",
                            )
                            ax.add_patch(rect)
                            ax.text(
                                x1, y1 - 3, label,
                                fontsize=11, color=color, fontweight="bold",
                                verticalalignment="bottom",
                                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor=color, linewidth=1),
                            )

                        ax.set_title(
                            f"样本 {idx + 1}: {len(detections)} 个检测框  (坐标格式: {coord_fmt}, 归一化: {coord_nrm})",
                            fontsize=13,
                        )
                        ax.axis("off")
                        plt.tight_layout()
                        plt.show()
                    else:
                        ui.show_warning(f"图像文件未找到: {img_path} (已尝试多种路径)")
                elif not PIL_AVAILABLE or not MATPLOTLIB_AVAILABLE:
                    ui.show_info("PIL/matplotlib不可用，仅展示数据格式详情（跳过图像绘制）")
                else:
                    ui.show_warning("样本中无图像路径")
else:
    ui.show_warning("无训练集数据，无法展示可视化")


## 总结

本Notebook完成了LabelMe标注数据的完整处理流程：

1. **标注清洗** → 验证完整性、过滤无效数据、识别重复标注
2. **统计分析** → 类别分布、标注数量、可视化报告
3. **均衡选择** → 基于类别的均衡采样
4. **格式转换** → Unsloth兼容格式 + 数据集划分
5. **输出验证** → JSONL格式完整性验证

> 📌 后续步骤：使用生成的JSONL文件训练Unsloth模型，根据统计报告调整数据策略

In [ ]:
if cleaning_result and stats_result and selection_result and conversion_result:
    ui.show_final_summary(cleaning_result, stats_result, selection_result, conversion_result, config)
else:
    ui.show_warning("部分步骤执行失败，请检查上方错误信息后重新运行对应单元格")